# Phase 0 — Data Collection
### Predicting Diabetes Risk Using Machine Learning
---

**Project**: End-to-end ML pipeline for diabetes risk prediction  
**Data Source**: CDC Behavioral Risk Factor Surveillance System (BRFSS)  
**Notebook Role**: Extract raw survey data from fixed-width ASCII files and save structured CSVs  
**Author**: Kota  
**Last Updated**: 2026-04

---

## Objective

This notebook handles the **data acquisition step** of the pipeline.

The CDC publishes BRFSS data annually as fixed-width ASCII files (.ASC), which cannot
be loaded directly into pandas without knowing each variable's exact column position.
This notebook automates the extraction by:

1. Parsing each year's HTML codebook to retrieve column positions programmatically
2. Using those positions to read only the relevant variables from the raw ASCII file
3. Saving clean, analysis-ready CSVs for downstream notebooks

**Why automate the codebook parsing?**  
Column positions shift slightly between survey years as the questionnaire evolves.
Hard-coding positions would break when a new year is added.
Parsing the codebook programmatically makes the pipeline robust across years
and eliminates manual maintenance.

---

## Dataset: CDC BRFSS

The Behavioral Risk Factor Surveillance System (BRFSS) is the world's largest
continuously conducted telephone health survey, operated by the U.S. Centers
for Disease Control and Prevention (CDC).

| Attribute | Detail |
|-----------|--------|
| Coverage | All 50 U.S. states + territories |
| Sample size | ~440,000–460,000 respondents per year |
| Collection method | Telephone survey (landline + cell) |
| Data format | Fixed-width ASCII (.ASC) |
| License | Public domain (CDC open data) |

**Why BRFSS over the commonly-used Pima Indians Diabetes Dataset?**

| | Pima Indians | BRFSS (this project) |
|---|---|---|
| Sample size | 768 | ~1.3 million (3 years) |
| Demographics | Pima Indian women only | Nationally representative |
| Variables | 8 | 22 (lifestyle, socioeconomic, clinical) |
| Recency | 1990s | 2022–2024 |

BRFSS provides real-world, large-scale data that better reflects the diversity
of the U.S. population and supports more generalisable model findings.

---

## Years Selected: 2022, 2023, 2024

| Year | Reason for inclusion |
|------|----------------------|
| 2022 | First post-COVID stable year with full state coverage |
| 2023 | Most recent year with fully released codebook |
| 2024 | Latest available data |

2020 and 2021 were excluded due to COVID-19 disruption to survey collection
and reporting patterns, which would introduce non-representative noise.


## 1. Imports and Environment

In [34]:
import re
import os
import pandas as pd
from pathlib import Path

print(f"pandas  : {pd.__version__}")
print(f"Working directory: {os.getcwd()}")


pandas  : 2.0.3
Working directory: d:\Users\kotae\Documents\Portfolio\project\Project 2\diabetes-risk-prediction\notebooks


## 2. Path Configuration

Raw files are stored in `data/source/` (not tracked by Git due to size).  
Processed outputs are written to `data/raw/` for use in downstream notebooks.

> **Reproducibility note**: To reproduce this notebook, download the BRFSS
> ASCII files and HTML codebooks for 2022–2024 from:  
> https://www.cdc.gov/brfss/annual_data/annual_data.htm  
> and place them in `data/source/`.


In [35]:
SOURCE_DIR = Path("../data/source")
OUTPUT_DIR = Path("../data/raw")

YEAR_FILES = {
    2022: {
        "asc":           SOURCE_DIR / "LLCP2022.ASC",
        "html":          SOURCE_DIR / "USCODE22_LLCP_102523.HTML",
        "record_length": 2052,
    },
    2023: {
        "asc":           SOURCE_DIR / "LLCP2023.ASC",
        "html":          SOURCE_DIR / "USCODE23_LLCP_021924.HTML",
        "record_length": 2111,
    },
    2024: {
        "asc":           SOURCE_DIR / "LLCP2024.ASC",
        "html":          SOURCE_DIR / "USCODE24_LLCP_082125.HTML",
        "record_length": 2111,
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source : {SOURCE_DIR.resolve()}")
print(f"Output : {OUTPUT_DIR.resolve()}")

print("\nFile availability check:")
all_ok = True
for year, cfg in YEAR_FILES.items():
    asc_ok  = cfg["asc"].exists()
    html_ok = cfg["html"].exists()
    print(f"  {year}  ASC: {'OK' if asc_ok else 'MISSING':<8}  HTML: {'OK' if html_ok else 'MISSING'}")
    if not (asc_ok and html_ok):
        all_ok = False
print("\nAll files present." if all_ok else "\nWARNING: some files are missing.")


Source : D:\Users\kotae\Documents\Portfolio\project\Project 2\diabetes-risk-prediction\data\source
Output : D:\Users\kotae\Documents\Portfolio\project\Project 2\diabetes-risk-prediction\data\raw

File availability check:
  2022  ASC: OK        HTML: OK
  2023  ASC: OK        HTML: OK
  2024  ASC: OK        HTML: OK

All files present.


## 3. Target Variable Definitions

### Variable selection rationale

Variables were selected based on established diabetes risk factors from clinical
literature and public health research. The selection covers five domains:

| Domain | Variables |
|--------|-----------|
| Target | `DIABETE4` |
| Physical health | `GENHLTH`, `PHYSHLTH`, `MENTHLTH`, `POORHLTH`, `CHECKUP1` |
| Lifestyle | `EXERANY2`, `DIFFWALK` |
| Medical history | `BPHIGH6`, `CVDINFR4`, `CVDSTRK3` |
| Demographics | `SEXVAR`, `EDUCA`, `INCOME3`, `_STATE` |
| Calculated variables | `_SEX`, `_RACE`, `_AGEG5YR`, `_BMI5CAT`, `_SMOKER3`, `_CHOLCH3` |

Calculated variables (prefixed with `_`) are derived by the CDC from raw survey
responses and provide standardised, analysis-ready encodings.

### Explicit width specification

Each variable is assigned an **explicit character width** rather than estimating
width from the gap to the next variable in the codebook.

**Why this matters**: In 2022, the variable immediately following `DIABETE4`
in the codebook differs from 2023 and 2024. Gap-based width estimation produced
a width of 3 for 2022, causing `DIABETE4` to read across adjacent columns and
return values in the range 100–356 — impossible under the BRFSS response schema
(valid codes: 1, 2, 3, 4, 7, 9). Explicit widths eliminate this cross-year
fragility entirely.


In [36]:
# Each entry: variable_name -> (description, character_width)
# Width = number of fixed-width characters to read for this variable.
# Single-choice responses = 1 char; day counts and multi-digit codes = 2 chars.
TARGET_VARIABLES = {
    # --- Target ---
    "DIABETE4": ("Ever told you had diabetes (target variable)",        1),
    "PREDIAB2": ("Pre-diabetes or borderline diabetes",                 1),

    # --- Physical health ---
    "GENHLTH":  ("General health status (1=Excellent to 5=Poor)",       1),
    "PHYSHLTH": ("Days physical health not good, past 30 days",         2),
    "MENTHLTH": ("Days mental health not good, past 30 days",           2),
    "POORHLTH": ("Days poor health limited usual activities",            2),
    "CHECKUP1": ("Time since last routine checkup",                      1),

    # --- Lifestyle ---
    "EXERANY2": ("Any physical activity in past 30 days",                1),
    "DIFFWALK": ("Difficulty walking or climbing stairs",                1),

    # --- Medical history ---
    "BPHIGH6":  ("Ever told blood pressure was high",                    1),
    "CVDINFR4": ("Ever told had a heart attack",                         1),
    "CVDSTRK3": ("Ever told had a stroke",                               1),

    # --- Demographics ---
    "SEXVAR":   ("Sex of respondent",                                    1),
    "EDUCA":    ("Highest education level completed",                    1),
    "INCOME3":  ("Annual household income",                              2),
    "_STATE":   ("State FIPS code",                                      2),

    # --- Calculated variables (CDC-derived) ---
    "_SEX":     ("Sex (calculated)",                                     1),
    "_RACE":    ("Race/ethnicity (calculated)",                          1),
    "_AGEG5YR": ("Age group in 5-year intervals (calculated)",           2),
    "_BMI5CAT": ("BMI category (calculated)",                            1),
    "_SMOKER3": ("Smoking status (calculated)",                          1),
    "_CHOLCH3": ("Cholesterol check within past 5 years (calculated)",   1),
}

print(f"{'Variable':<15} {'Width':>5}  Description")
print("-" * 65)
for name, (desc, width) in TARGET_VARIABLES.items():
    print(f"  {name:<13} {width:>5}  {desc}")
print(f"\nTotal: {len(TARGET_VARIABLES)} variables")


Variable        Width  Description
-----------------------------------------------------------------
  DIABETE4          1  Ever told you had diabetes (target variable)
  PREDIAB2          1  Pre-diabetes or borderline diabetes
  GENHLTH           1  General health status (1=Excellent to 5=Poor)
  PHYSHLTH          2  Days physical health not good, past 30 days
  MENTHLTH          2  Days mental health not good, past 30 days
  POORHLTH          2  Days poor health limited usual activities
  CHECKUP1          1  Time since last routine checkup
  EXERANY2          1  Any physical activity in past 30 days
  DIFFWALK          1  Difficulty walking or climbing stairs
  BPHIGH6           1  Ever told blood pressure was high
  CVDINFR4          1  Ever told had a heart attack
  CVDSTRK3          1  Ever told had a stroke
  SEXVAR            1  Sex of respondent
  EDUCA             1  Highest education level completed
  INCOME3           2  Annual household income
  _STATE            2  State 

## 4. Codebook Parser

Each year's HTML codebook contains one entry per variable with the format:

```
Column: 149
Type of Variable: Num
SAS Variable Name: DIABETE4
```

The parser extracts all `{variable_name: column_number}` pairs using a
single regular expression. Column numbers are 1-indexed in the codebook
and converted to 0-indexed for use with `pd.read_fwf()`.


In [37]:
def parse_codebook(html_path: Path) -> dict:
    """
    Parse a BRFSS HTML codebook and return {variable_name: start_column}.
    Column numbers are 1-indexed as published by the CDC.
    """
    with open(html_path, encoding="windows-1252") as f:
        content = f.read()
    pattern = r"Column:&nbsp;(\d+).*?SAS&nbsp;Variable&nbsp;Name:&nbsp;([A-Z_][A-Z0-9_]*)"
    matches = re.findall(pattern, content)
    return {name: int(col) for col, name in matches}


def build_colspecs(var_col: dict, target_vars: dict) -> tuple:
    """
    Build colspecs list for pd.read_fwf() using explicit variable widths.

    Parameters
    ----------
    var_col     : {variable_name: start_column (1-indexed)} from codebook
    target_vars : {variable_name: (description, explicit_width)}

    Returns
    -------
    names    : list of variable names
    colspecs : list of (start, end) tuples — 0-indexed, half-open intervals
    """
    names, colspecs, missing = [], [], []

    for var, (desc, width) in target_vars.items():
        if var not in var_col:
            missing.append(var)
            continue
        start = var_col[var] - 1   # convert 1-indexed → 0-indexed
        names.append(var)
        colspecs.append((start, start + width))

    if missing:
        print(f"  NOTE — variables not found in this year's codebook: {missing}")
        print("  These will appear as all-NaN in the output.")
        print("  Investigation and remediation planned for Phase 2 (Data Cleaning).")

    return names, colspecs


# Verification: confirm DIABETE4 reads as exactly 1 character in 2023
print("── Codebook parser verification (2023) ──")
var_col_2023 = parse_codebook(YEAR_FILES[2023]["html"])
print(f"  Variables detected in codebook: {len(var_col_2023)}")

names_test, colspecs_test = build_colspecs(var_col_2023, TARGET_VARIABLES)
print(f"\n  {'Variable':<15} {'Start':>7}  {'End':>7}  {'Width':>6}")
print("  " + "-" * 38)
for name, (s, e) in zip(names_test, colspecs_test):
    print(f"  {name:<15} {s:>7}  {e:>7}  {e-s:>6}")


── Codebook parser verification (2023) ──
  Variables detected in codebook: 344

  Variable          Start      End   Width
  --------------------------------------
  DIABETE4            148      149       1
  PREDIAB2            260      261       1
  GENHLTH             100      101       1
  PHYSHLTH            101      103       2
  MENTHLTH            103      105       2
  POORHLTH            105      107       2
  CHECKUP1            111      112       1
  EXERANY2            112      113       1
  DIFFWALK            217      218       1
  BPHIGH6             132      133       1
  CVDINFR4            137      138       1
  CVDSTRK3            139      140       1
  SEXVAR               87       88       1
  EDUCA               186      187       1
  INCOME3             203      205       2
  _STATE                0        2       2
  _SEX               2063     2064       1
  _RACE              2059     2060       1
  _AGEG5YR           2064     2066       2
  _BMI5CAT        

## 5. ASC File Loader

The loader follows a three-step process per year:

1. **Parse codebook** → retrieve column positions from the HTML file
2. **Build colspecs** → map variable names to `(start, end)` character positions
3. **Read ASCII file** → use `pd.read_fwf()` to extract only the required columns

All values are initially read as strings to prevent pandas from silently
misinterpreting BRFSS special codes (e.g. `77` = "Don't know", `99` = "Refused")
as numeric outliers. Conversion to numeric is applied after loading.


In [38]:
def load_brfss_asc(year: int, year_config: dict, target_vars: dict) -> pd.DataFrame:
    """
    Load one year of BRFSS data from an ASCII fixed-width file.

    Returns a DataFrame with a YEAR column prepended,
    all target variables as float64, and missing/invalid codes preserved as NaN.
    """
    asc_path  = year_config["asc"]
    html_path = year_config["html"]

    print(f"\n{'=' * 60}")
    print(f"  Processing {year}")
    print(f"{'=' * 60}")

    # Step 1: parse codebook
    print(f"  [1/3] Parsing codebook ... {html_path.name}")
    var_col = parse_codebook(html_path)
    print(f"        {len(var_col)} variables in codebook")

    # Step 2: build colspecs with explicit widths
    names, colspecs = build_colspecs(var_col, target_vars)
    print(f"  [2/3] {len(names)} variables mapped to colspecs")

    if "DIABETE4" in names:
        idx = names.index("DIABETE4")
        s, e = colspecs[idx]
        print(f"        DIABETE4 colspec: ({s}, {e})  width = {e - s}  [expected: 1]")

    # Step 3: read ASCII file
    file_mb = asc_path.stat().st_size / 1024 / 1024
    print(f"  [3/3] Reading {asc_path.name} ({file_mb:.0f} MB) ...")

    df = pd.read_fwf(
        asc_path,
        colspecs=colspecs,
        names=names,
        dtype=str,          # read as string first to preserve all codes
        encoding="latin-1",
    )

    # Convert to numeric; invalid codes and blanks become NaN
    for col in df.columns:
        df[col] = pd.to_numeric(df[col].str.strip(), errors="coerce")

    # For variables absent from this year's codebook, add NaN column
    for var in target_vars:
        if var not in df.columns:
            df[var] = float("nan")

    # Reorder columns to match TARGET_VARIABLES definition order
    df = df[[v for v in target_vars if v in df.columns]]

    # Prepend YEAR identifier
    df.insert(0, "YEAR", year)

    print(f"  Done  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
    return df


## 6. Execute: Process All Three Years

Each year is processed independently and saved as an individual CSV.
A combined 3-year dataset is also produced as the primary input for
downstream analysis notebooks.


In [39]:
all_years = {}

for year, config in YEAR_FILES.items():
    df = load_brfss_asc(year, config, TARGET_VARIABLES)
    all_years[year] = df

    out_path = OUTPUT_DIR / f"brfss_{year}_diabetes.csv"
    df.to_csv(out_path, index=False)
    mb = out_path.stat().st_size / 1024 / 1024
    print(f"  Saved: {out_path.name}  ({mb:.1f} MB)")

print("\nAll years processed successfully.")



  Processing 2022
  [1/3] Parsing codebook ... USCODE22_LLCP_102523.HTML
        324 variables in codebook
  NOTE — variables not found in this year's codebook: ['BPHIGH6', '_RACE', '_CHOLCH3']
  These will appear as all-NaN in the output.
  Investigation and remediation planned for Phase 2 (Data Cleaning).
  [2/3] 19 variables mapped to colspecs
        DIABETE4 colspec: (128, 129)  width = 1  [expected: 1]
  [3/3] Reading LLCP2022.ASC (872 MB) ...
  Done  →  445,132 rows  ×  23 columns
  Saved: brfss_2022_diabetes.csv  (31.7 MB)

  Processing 2023
  [1/3] Parsing codebook ... USCODE23_LLCP_021924.HTML
        344 variables in codebook
  [2/3] 22 variables mapped to colspecs
        DIABETE4 colspec: (148, 149)  width = 1  [expected: 1]
  [3/3] Reading LLCP2023.ASC (873 MB) ...
  Done  →  433,323 rows  ×  23 columns
  Saved: brfss_2023_diabetes.csv  (33.9 MB)

  Processing 2024
  [1/3] Parsing codebook ... USCODE24_LLCP_082125.HTML
        297 variables in codebook
  NOTE — variables

## 7. Create Combined Dataset (2022–2024)

The three annual datasets are concatenated into a single file.
The `YEAR` column allows year-level filtering and cross-year comparisons
in downstream analyses.


In [40]:
df_all = pd.concat(all_years.values(), ignore_index=True)

print(f"Combined dataset: {df_all.shape[0]:,} rows  ×  {df_all.shape[1]} columns")
print("\nRows per year:")
print(df_all["YEAR"].value_counts().sort_index().to_string())

combined_path = OUTPUT_DIR / "brfss_2022_2024_combined.csv"
df_all.to_csv(combined_path, index=False)
mb = combined_path.stat().st_size / 1024 / 1024
print(f"\nSaved: {combined_path.name}  ({mb:.1f} MB)")


Combined dataset: 1,336,125 rows  ×  23 columns

Rows per year:
YEAR
2022    445132
2023    433323
2024    457670

Saved: brfss_2022_2024_combined.csv  (100.4 MB)


## 8. Output Validation

### Target variable: DIABETE4

BRFSS response codes for `DIABETE4`:

| Code | Meaning |
|------|---------|
| 1 | Yes — diagnosed with diabetes |
| 2 | Yes — but only during pregnancy |
| 3 | No |
| 4 | No — pre-diabetes or borderline |
| 7 | Don't know / Not sure |
| 9 | Refused |

Any value outside {1, 2, 3, 4, 7, 9} indicates a colspec error.
The expected mean across years is approximately 2.7, reflecting that
the majority of respondents (code 3) have not been diagnosed.

> **Note on 2022 codebook discrepancies**: In 2022, three calculated variables
> (`BPHIGH6`, `_RACE`, `_CHOLCH3`) were not found under their expected names
> in the codebook. These columns are present but all-NaN for 2022.
> Investigation and remediation are planned for **Phase 2 (Data Cleaning)**.


In [41]:
VALID_DIABETE4 = {1, 2, 3, 4, 7, 9}

print("── DIABETE4 distribution by year ──\n")

all_valid = True
for year, df in all_years.items():
    unique_vals = set(df["DIABETE4"].dropna().astype(int).unique())
    invalid     = unique_vals - VALID_DIABETE4
    mean_val    = df["DIABETE4"].mean()
    status      = "PASS" if not invalid else f"FAIL — unexpected values: {sorted(invalid)}"
    if invalid:
        all_valid = False

    print(f"[{year}]  mean = {mean_val:.3f}  unique values = {sorted(unique_vals)}  → {status}")
    counts = df["DIABETE4"].value_counts(dropna=False).sort_index()
    total  = len(df)
    for val, cnt in counts.items():
        flag = "  <-- INVALID" if (not __import__('math').isnan(float(val)) and int(val) not in VALID_DIABETE4) else ""
        print(f"  {str(val):>6} : {cnt:>8,}  ({cnt/total*100:5.1f}%){flag}")
    print()

result = "All years PASS validation." if all_valid else "VALIDATION FAILED — review colspecs."
print(result)


── DIABETE4 distribution by year ──

[2022]  mean = 2.751  unique values = [1, 2, 3, 4, 7, 9]  → PASS
     1.0 :   61,158  ( 13.7%)
     2.0 :    3,836  (  0.9%)
     3.0 :  368,722  ( 82.8%)
     4.0 :   10,329  (  2.3%)
     7.0 :      763  (  0.2%)
     9.0 :      321  (  0.1%)
     nan :        3  (  0.0%)

[2023]  mean = 2.751  unique values = [1, 2, 3, 4, 7, 9]  → PASS
     1.0 :   59,786  ( 13.8%)
     2.0 :    3,253  (  0.8%)
     3.0 :  358,706  ( 82.8%)
     4.0 :   10,594  (  2.4%)
     7.0 :      683  (  0.2%)
     9.0 :      296  (  0.1%)
     nan :        5  (  0.0%)

[2024]  mean = 2.740  unique values = [1, 2, 3, 4, 7, 9]  → PASS
     1.0 :   65,809  ( 14.4%)
     2.0 :    3,395  (  0.7%)
     3.0 :  376,125  ( 82.2%)
     4.0 :   11,307  (  2.5%)
     7.0 :      798  (  0.2%)
     9.0 :      232  (  0.1%)
     nan :        4  (  0.0%)

All years PASS validation.


In [42]:
# Cross-year mean comparison (expected: ~2.7 for all years)
print("── Cross-year sanity check ──")
print("Expected DIABETE4 mean ≈ 2.7 (driven by majority code-3 respondents)\n")
for year, df in all_years.items():
    m    = df["DIABETE4"].mean()
    flag = "" if 2.0 <= m <= 3.5 else "  <-- SUSPICIOUS"
    print(f"  {year}  mean = {m:.4f}{flag}")


── Cross-year sanity check ──
Expected DIABETE4 mean ≈ 2.7 (driven by majority code-3 respondents)

  2022  mean = 2.7510
  2023  mean = 2.7514
  2024  mean = 2.7397


In [43]:
# Missing value overview across all years
print("── Missing value summary ──\n")
for year, df in all_years.items():
    miss     = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(1)
    flagged  = miss_pct[miss_pct > 5].index.tolist()
    print(f"[{year}]  Variables with >5% missing: {flagged if flagged else 'None'}")

print("\nDetailed 2023 missing rates:")
df23     = all_years[2023]
miss_pct = (df23.isnull().sum() / len(df23) * 100).round(1).sort_values(ascending=False)
print(miss_pct[miss_pct > 0].to_string())


── Missing value summary ──

[2022]  Variables with >5% missing: ['PREDIAB2', 'POORHLTH', 'BPHIGH6', '_RACE', '_BMI5CAT', '_CHOLCH3']
[2023]  Variables with >5% missing: ['PREDIAB2', 'POORHLTH', '_BMI5CAT']
[2024]  Variables with >5% missing: ['PREDIAB2', 'POORHLTH', 'BPHIGH6', '_BMI5CAT', '_CHOLCH3']

Detailed 2023 missing rates:
PREDIAB2    61.7
POORHLTH    41.8
_BMI5CAT     9.4
DIFFWALK     3.7
INCOME3      1.9


In [44]:
# Preview
print("── Combined dataset — first 5 rows ──")
df_all.head()


── Combined dataset — first 5 rows ──


,YEAR,DIABETE4,PREDIAB2,GENHLTH,PHYSHLTH,MENTHLTH,POORHLTH,CHECKUP1,EXERANY2,DIFFWALK,...,SEXVAR,EDUCA,INCOME3,_STATE,_SEX,_RACE,_AGEG5YR,_BMI5CAT,_SMOKER3,_CHOLCH3
0,2022,1.0,NaN,2.0,88.0,88.0,NaN,1.0,2.0,2.0,...,2,6.0,99.0,1,2,NaN,13,NaN,4,NaN
1,2022,3.0,NaN,1.0,88.0,88.0,NaN,8.0,2.0,2.0,...,2,4.0,5.0,1,2,NaN,13,3.0,4,NaN
2,2022,3.0,NaN,2.0,2.0,3.0,2.0,1.0,1.0,2.0,...,2,6.0,10.0,1,2,NaN,8,3.0,4,NaN
3,2022,3.0,NaN,1.0,88.0,88.0,NaN,1.0,1.0,2.0,...,2,4.0,77.0,1,2,NaN,14,2.0,2,NaN
4,2022,3.0,NaN,4.0,2.0,88.0,88.0,1.0,1.0,2.0,...,2,5.0,5.0,1,2,NaN,5,2.0,4,NaN


## 9. Summary and Key Decisions

### What was accomplished

| Step | Output |
|------|--------|
| Parsed HTML codebooks (3 years) | Variable → column position mapping |
| Extracted 22 variables from ASCII files | ~440,000 rows × 23 cols per year |
| Validated DIABETE4 codes | All years: values in {1,2,3,4,7,9} ✅ |
| Saved individual year CSVs | `brfss_2022/2023/2024_diabetes.csv` |
| Saved combined dataset | `brfss_2022_2024_combined.csv` (~1.3M rows) |

### Key engineering decisions

| Decision | Rationale |
|----------|-----------|
| Parse codebook programmatically | Eliminates manual column mapping; robust across years |
| Explicit variable widths | Prevents cross-year colspec errors (fixed 2022 DIABETE4 bug) |
| Read as string, convert to numeric | Preserves BRFSS special codes (77, 99) for proper handling in Phase 2 |
| Store NaN for absent 2022 variables | Avoids silent data corruption; flagged for Phase 2 investigation |
| Exclude 2020–2021 | COVID-19 disruption to survey patterns |

### Known issues for Phase 2

| Issue | Variables affected | Year |
|-------|-------------------|------|
| Variable name change across years | `BPHIGH6`, `_RACE`, `_CHOLCH3` | 2022 only |
| High missing rate | `PREDIAB2` (~62%), `POORHLTH` (~42%) | All years |
| Special codes not yet handled | 7, 9, 77, 88, 99 (Don't know / Refused) | All years |

---

**Next step**: `01_data_understanding.ipynb` — exploratory data analysis,
distribution visualisations, and missing value profiling across all 22 variables.
